In [1]:
import pandas as pd
from pathlib import Path
processed_path = Path("../data/processed")

import sys
sys.path.insert(0, str(Path("../scripts").resolve()))


In [2]:
data_path = Path(
    "/mnt/c/Users/vetts/Downloads/MimicIII/mimic-iii-clinical-database-1.4"
)

In [3]:
lab_df = pd.read_parquet(
    processed_path / "lab_features_first24h.parquet"
)

vital_df = pd.read_parquet(
    processed_path / "vital_features_first24h.parquet"
)

output_df = pd.read_parquet(
    processed_path / "output_features_first24h.parquet"
)

intervention_df = pd.read_parquet(
    processed_path / "intervention_features_first24h.parquet"
)

demographic_df = pd.read_parquet(
    processed_path / "demographic_features.parquet"
)

In [4]:
print("Lab:", lab_df.shape)
print("Vital:", vital_df.shape)
print("Output:", output_df.shape)
print("Intervention:", intervention_df.shape)
print("Demographic:", demographic_df.shape)

Lab: (44626, 39)
Vital: (44549, 33)
Output: (45253, 5)
Intervention: (45253, 4)
Demographic: (45253, 5)


In [5]:
cohort_ids = demographic_df[["ICUSTAY_ID"]].copy()

cohort_ids["has_lab"] = cohort_ids["ICUSTAY_ID"].isin(
    lab_df["ICUSTAY_ID"]
)

cohort_ids["has_vital"] = cohort_ids["ICUSTAY_ID"].isin(
    vital_df["ICUSTAY_ID"]
)

cohort_ids.value_counts(
    ["has_lab", "has_vital"]
)

has_lab  has_vital
True     True         44187
         False          439
False    True           362
         False          265
Name: count, dtype: int64

In [6]:
cohort_df = pd.read_parquet(processed_path / "adult_icu_cohort_first24h.parquet")
admissions_df = pd.read_csv(
    data_path / "ADMISSIONS.csv" / "ADMISSIONS.csv",
    usecols=["SUBJECT_ID", "HADM_ID", "HOSPITAL_EXPIRE_FLAG"]
)
assert admissions_df["HADM_ID"].is_unique


In [7]:
mortality_df = cohort_df[["ICUSTAY_ID", "SUBJECT_ID", "HADM_ID"]].merge(
    admissions_df, on=["SUBJECT_ID", "HADM_ID"], how="left", validate="many_to_one"
)
assert mortality_df["HOSPITAL_EXPIRE_FLAG"].notna().all()
cohort_missing_df = cohort_ids.merge(
    mortality_df[["ICUSTAY_ID", "HOSPITAL_EXPIRE_FLAG"]],
    on="ICUSTAY_ID", how="left", validate="one_to_one"
)


In [8]:
cohort_missing_df.groupby(
    ["has_lab", "has_vital"]
)["HOSPITAL_EXPIRE_FLAG"].agg(["count", "sum", "mean"])

count   sum      mean
has_lab has_vital                       
False   False        265     5  0.018868
        True         362    82  0.226519
True    False        439    54  0.123007
        True       44187  5219  0.118112

In [9]:
final_df = cohort_df[["ICUSTAY_ID"]].copy()

final_df["lab_missing"] = (
    ~final_df["ICUSTAY_ID"].isin(lab_df["ICUSTAY_ID"])
).astype(int)

final_df["vital_missing"] = (
    ~final_df["ICUSTAY_ID"].isin(vital_df["ICUSTAY_ID"])
).astype(int)

In [10]:
print(final_df["lab_missing"].value_counts())
print(final_df["vital_missing"].value_counts())

lab_missing
0    44626
1      627
Name: count, dtype: int64
vital_missing
0    44549
1      704
Name: count, dtype: int64


In [11]:
final_df = final_df.merge(
    lab_df,
    on="ICUSTAY_ID",
    how="left", validate="one_to_one"
)

In [12]:
final_df.shape

(45253, 41)

In [13]:
final_df = final_df.merge(
    vital_df,
    on="ICUSTAY_ID",
    how="left", validate="one_to_one"
)

In [14]:
final_df.shape

(45253, 73)

In [15]:
for df in [output_df, intervention_df, demographic_df]:
    final_df = final_df.merge(
        df,
        on="ICUSTAY_ID",
        how="left", validate="one_to_one"
    )

In [16]:
final_df.shape

(45253, 84)

In [17]:
final_df["ICUSTAY_ID"].duplicated().sum()

np.int64(0)

In [18]:
missing_final = final_df.isna().sum()

missing_final[missing_final > 0].sort_values(ascending=False)

ebl_24h             40962
Lactate_change      24771
Lactate_first       24771
GCS Total_first     20274
GCS Total_max       20274
                    ...  
SpO2_last             726
Heart Rate_last       709
Heart Rate_first      709
Heart Rate_min        709
Heart Rate_max        709
Length: 72, dtype: int64

In [19]:
final_df.loc[final_df["ebl_present"].eq(0), "ebl_24h"] = 0
assert final_df.loc[final_df["ebl_present"].eq(1), "ebl_24h"].notna().all()

In [20]:
final_df["ebl_24h"].isna().sum()

np.int64(0)

In [21]:
final_df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

['gender', 'admission_type', 'admission_location']

In [22]:
final_df = final_df.merge(
    mortality_df[["ICUSTAY_ID", "HOSPITAL_EXPIRE_FLAG"]],
    on="ICUSTAY_ID",
    how="left", validate="one_to_one"
)

In [23]:
print(final_df["HOSPITAL_EXPIRE_FLAG"].isna().sum())

print(
    final_df["HOSPITAL_EXPIRE_FLAG"].value_counts()
)

print(
    final_df["HOSPITAL_EXPIRE_FLAG"].value_counts(normalize=True) * 100
)

0
HOSPITAL_EXPIRE_FLAG
0    39893
1     5360
Name: count, dtype: int64
HOSPITAL_EXPIRE_FLAG
0    88.155481
1    11.844519
Name: proportion, dtype: float64


In [24]:
leakage_keywords = [
    "death",
    "expire",
    "discharge",
    "dod",
    "deathtime",
    "outtime"
]

[
    col for col in final_df.columns
    if any(word in col.lower() for word in leakage_keywords)
]

['HOSPITAL_EXPIRE_FLAG']

In [25]:
import numpy as np

print("Infinite values:", np.isinf(
    final_df.select_dtypes(include="number")
).sum().sum())

print("Age range:",
      final_df["age"].min(),
      final_df["age"].max())

print("Negative urine:",
      (final_df["urine_output_24h"] < 0).sum())

print("Negative EBL:",
      (final_df["ebl_24h"] < 0).sum())

print("Target values:",
      final_df["HOSPITAL_EXPIRE_FLAG"].unique())

print("Lab missing values:",
      final_df["lab_missing"].unique())

print("Vital missing values:",
      final_df["vital_missing"].unique())

Infinite values: 0
Age range: 18.021689894035035 90.0
Negative urine: 0
Negative EBL: 0
Target values: [1 0]
Lab missing values: [0 1]
Vital missing values: [0 1]


In [26]:
output_df = pd.read_parquet(
    processed_path / "output_features_first24h.parquet"
)

final_df = cohort_df[["ICUSTAY_ID"]].copy()

final_df["lab_missing"] = (
    ~final_df["ICUSTAY_ID"].isin(lab_df["ICUSTAY_ID"])
).astype(int)

final_df["vital_missing"] = (
    ~final_df["ICUSTAY_ID"].isin(vital_df["ICUSTAY_ID"])
).astype(int)

for df in [
    lab_df,
    vital_df,
    output_df,
    intervention_df,
    demographic_df
]:
    final_df = final_df.merge(
        df,
        on="ICUSTAY_ID",
        how="left", validate="one_to_one"
    )

final_df.loc[final_df["ebl_present"].eq(0), "ebl_24h"] = 0
assert final_df.loc[final_df["ebl_present"].eq(1), "ebl_24h"].notna().all()

final_df = final_df.merge(
    mortality_df[["ICUSTAY_ID", "HOSPITAL_EXPIRE_FLAG"]],
    on="ICUSTAY_ID",
    how="left", validate="one_to_one"
)

final_df.shape

(45253, 85)

In [27]:
(final_df["urine_output_24h"] < 0).sum()

np.int64(0)

In [28]:
non_negative_cols = [
    col for col in final_df.select_dtypes(include="number").columns
    if not col.endswith("_change")
]

negative_counts = {
    col: int((final_df[col] < 0).sum())
    for col in non_negative_cols
}

{
    col: count
    for col, count in negative_counts.items()
    if count > 0
}

{'Anion Gap_first': 1}

## Pre-model data-quality review (2026-09-12)

Corrections were made before source aggregation, not patched into final feature values:
- Notebook 02 applies broad [MIMIC-code vital bounds](https://github.com/MIT-LCP/mimic-code/blob/main/mimic-iii/concepts/firstday/vitals_first_day.sql), Fahrenheit validation before Celsius conversion, integer GCS 3–15, and exclusion of ERROR-marked measurements. Source group keys remain present, preserving the existing `vital_missing` row-presence definition. Individual NaNs are not filled.
- Notebook 01 invalidates nonpositive hemoglobin/INR and nonfinite lab values before first/last/change aggregation. Only 6 existing feature cells changed; the uncleaned raw-data reconstruction was checked against the old lab table before regeneration. Negative changes are legitimate. A [negative anion gap can occur](https://pmc.ncbi.nlm.nih.gov/articles/PMC9213217/), so the -4 observation is retained for review, not automatically erased.
- Notebook 03 rejects negative/nonfinite urine entries and gross individual entries above 100,000 mL before summation. The -1,500 mL entry for ICU 223940 remains excluded (corrected total 1,200 mL). The 555,975 mL entry for ICU 247320 is excluded (corrected total 5,215 mL). This is a broad gross-error rule, not a percentile-based cap. The 47,050 mL and 11,000 mL entries remain for manual review.
- EBL values are unchanged. Only `ebl_present == 0` implies final EBL=0; an unexpected missing EBL for `ebl_present == 1` now fails rather than being silently filled.
- Cohort IDs and feature lists are unchanged. No one is dropped for missing data. No category encoding, median imputation, scaling or outlier winsorization is performed.

### Remaining modeling considerations

Use patient-level (SUBJECT_ID-grouped) train/validation/test splits to avoid repeated patients appearing on both sides; obtain the group mapping separately from the cohort, not as a model feature. Fit imputation, category encoding, scaling and any learned outlier treatment on training data only. Target must be separated explicitly from predictors. EBL tails, remaining urine extremes, very high glucose/INR/electrolytes/cell counts and assay-rounded zero creatinine/BUN/WBC require review; they are not automatically impossible. Vital extraction bounds are broad quality screens, not normal ranges, and remaining extreme but accepted temperatures or rates merit sensitivity analysis.

This review checks structural leakage and the existing ICU-time filtering. It is not a clinical adjudication of every extreme value or a reconstruction of historical availability of every source field.


In [29]:
import sys
import json
from data_quality import numeric_audit, vital_valid

# Table integrity and exact feature schema; no duplicate-column suffixes or row multiplication.
source_tables = {"lab":lab_df, "vital":vital_df, "output":output_df,
                 "intervention":intervention_df, "demographic":demographic_df}
assert len(cohort_df) == cohort_df.ICUSTAY_ID.nunique() == 45253
for name, table in source_tables.items():
    assert table.ICUSTAY_ID.notna().all() and table.ICUSTAY_ID.is_unique, name
    assert set(table.ICUSTAY_ID).issubset(set(cohort_df.ICUSTAY_ID)), name
expected_columns = ["ICUSTAY_ID", "lab_missing", "vital_missing"]
for table in source_tables.values():
    expected_columns.extend(c for c in table.columns if c != "ICUSTAY_ID")
expected_columns.append("HOSPITAL_EXPIRE_FLAG")
assert len(expected_columns) == len(set(expected_columns))
assert final_df.columns.tolist() == expected_columns
assert len(final_df) == final_df.ICUSTAY_ID.nunique() == 45253
assert final_df.ICUSTAY_ID.equals(cohort_df.ICUSTAY_ID)
assert final_df.HOSPITAL_EXPIRE_FLAG.notna().all()
assert final_df.HOSPITAL_EXPIRE_FLAG.isin([0,1]).all()
assert final_df.lab_missing.equals((~final_df.ICUSTAY_ID.isin(lab_df.ICUSTAY_ID)).astype(int))
assert final_df.vital_missing.equals((~final_df.ICUSTAY_ID.isin(vital_df.ICUSTAY_ID)).astype(int))

# Target and identifiers are excluded from the predictor list.
model_feature_columns = [c for c in final_df if c not in ["ICUSTAY_ID", "HOSPITAL_EXPIRE_FLAG"]]
leakage_terms = ["death", "expire", "discharge", "dod", "outtime", "deathtime", "dischtime"]
leakage_columns = [c for c in model_feature_columns if any(term in c.lower() for term in leakage_terms)]
assert not leakage_columns, leakage_columns
assert not {"SUBJECT_ID","HADM_ID","INTIME","OUTTIME"}.intersection(model_feature_columns)
for name in ["lab_missing","vital_missing","chest_tube_present","ebl_present",
             "mechanical_ventilation_present","vasopressor_present","rrt_present"]:
    assert final_df[name].notna().all() and final_df[name].isin([0,1]).all(), name
assert final_df.loc[final_df.ebl_present.eq(0),'ebl_24h'].eq(0).all()
assert final_df.loc[final_df.ICUSTAY_ID.eq(223940),'urine_output_24h'].iloc[0] == 1200
assert final_df.loc[final_df.ICUSTAY_ID.eq(247320),'urine_output_24h'].iloc[0] == 5215

# Every vital summary obeys the source rules; first/last remain within min/max.
for name in {c.rsplit('_',1)[0] for c in vital_df if c != 'ICUSTAY_ID'}:
    for statistic in ['first','last','min','max']:
        values=final_df[f'{name}_{statistic}'].dropna()
        assert vital_valid(name,values).all(), (name,statistic)
    for statistic in ['first','last']:
        values=final_df[f'{name}_{statistic}']
        known=values.notna()
        assert values[known].ge(final_df.loc[known,f'{name}_min']).all()
        assert values[known].le(final_df.loc[known,f'{name}_max']).all()

numeric_quality = numeric_audit(final_df)
assert numeric_quality.infinite.sum() == 0
negative_exceptions = {c for c in final_df if c.endswith('_change')} | {'Anion Gap_first'}
invalid_negative = numeric_quality.loc[(numeric_quality.negative>0)&~numeric_quality.column.isin(negative_exceptions)]
assert invalid_negative.empty, invalid_negative
for name in ['Hemoglobin','INR(PT)']:
    assert final_df[name+'_first'].dropna().gt(0).all()
    assert (final_df[name+'_first']+final_df[name+'_change']).dropna().gt(0).all()
assert final_df.age.between(18,90).all()
for name in ['gender','admission_type','admission_location']:
    assert pd.api.types.is_string_dtype(final_df[name].dtype), name

# Missing values may remain only inside selected lab/vital columns and urine output.
expected_missing_columns = (set(lab_df.columns)|set(vital_df.columns)|{'urine_output_24h'})-{'ICUSTAY_ID'}
missing_counts=final_df.isna().sum()
missing_counts=missing_counts[missing_counts>0].sort_values(ascending=False)
assert set(missing_counts.index).issubset(expected_missing_columns)
print('Final shape:',final_df.shape,'| unique ICU IDs:',final_df.ICUSTAY_ID.nunique())
print('Predictors:',len(model_feature_columns),'| target: HOSPITAL_EXPIRE_FLAG (separate)')
print('Target distribution:',final_df.HOSPITAL_EXPIRE_FLAG.value_counts().to_dict())
print('Unexpected leakage columns:',leakage_columns)
print('Infinite values:',int(numeric_quality.infinite.sum()),'| unexpected negative columns: none')
print('Missing values by column (no imputation):')
print(missing_counts.to_string())
missing_by_group = {
    'lab':int(final_df[[c for c in lab_df if c!='ICUSTAY_ID']].isna().sum().sum()),
    'vital':int(final_df[[c for c in vital_df if c!='ICUSTAY_ID']].isna().sum().sum()),
    'urine_output_24h':int(final_df.urine_output_24h.isna().sum())}
print('Missing cells by group:',missing_by_group)
print('Rows with >=1 missing feature:',int(final_df.isna().any(axis=1).sum()))
print('Missing-table indicators:',{c:final_df[c].value_counts().to_dict() for c in ['lab_missing','vital_missing']})
print('Numeric audit (all numeric features; tails are flags, not automatic removals):')
print(numeric_quality.to_string(index=False))
print('EBL among present rows:',final_df.loc[final_df.ebl_present.eq(1),'ebl_24h'].quantile([.99,.995,.999,1]).to_dict())

report_path = processed_path.parent.parent / 'reports'
report_path.mkdir(exist_ok=True)
numeric_quality.to_csv(report_path/'data_quality_numeric_audit.csv',index=False)
missing_counts.rename('missing_count').to_csv(report_path/'data_quality_missing.csv')
quality_summary = {'shape':list(final_df.shape),'unique_ICUSTAY_ID':int(final_df.ICUSTAY_ID.nunique()),
    'predictor_count':len(model_feature_columns),'infinite_values':0,'unexpected_leakage_columns':leakage_columns,
    'missing_cells_by_group':missing_by_group,'total_missing_cells':int(final_df.isna().sum().sum()),
    'rows_with_missing':int(final_df.isna().any(axis=1).sum()),
    'missing_by_column':{k:int(v) for k,v in missing_counts.items()},
    'target_counts':{str(k):int(v) for k,v in final_df.HOSPITAL_EXPIRE_FLAG.value_counts().items()}}
(report_path/'data_quality_summary.json').write_text(json.dumps(quality_summary,indent=2)+'\n')
final_path=processed_path/'final_dataset.parquet'
partial_path=final_path.with_suffix('.parquet.partial')
final_df.to_parquet(partial_path,index=False)
pd.testing.assert_frame_equal(pd.read_parquet(partial_path),final_df)
partial_path.replace(final_path)
print('Saved validated final dataset:',final_path)


Final shape: (45253, 85) | unique ICU IDs: 45253
Predictors: 83 | target: HOSPITAL_EXPIRE_FLAG (separate)
Target distribution: {0: 39893, 1: 5360}
Unexpected leakage columns: []
Infinite values: 0 | unexpected negative columns: none
Missing values by column (no imputation):
Lactate_change              24771
Lactate_first               24771
GCS Total_last              20274
GCS Total_first             20274
GCS Total_min               20274
GCS Total_max               20274
pO2_change                  18552
pCO2_first                  18552
pCO2_change                 18552
pO2_first                   18552
pH_change                   17414
pH_first                    17414
INR(PT)_first                9344
INR(PT)_change               9344
Calcium, Total_first         7666
Calcium, Total_change        7666
Phosphate_first              7424
Phosphate_change             7424
Mean BP_first                6725
Mean BP_max                  6725
Mean BP_last                 6725
Mean BP_min

## Retained clinical review flags
Additional reference: [MIMIC-III benchmark ranges](https://github.com/YerevaNN/mimic3-benchmarks/blob/master/mimic3benchmark/resources/variable_ranges.csv). Only review flags are derived from these ranges; their imputation and clipping policies are not used. The calcium and urine thresholds below are explicitly conservative review heuristics. Retained flagged values must not be described as clinically validated.


In [30]:
# Review flags are diagnostics, not new model features and not automatic deletion rules.
review_rows = []
def review(column, mask, reason):
    count = int(mask.fillna(False).sum())
    if count:
        review_rows.append({'column': column, 'flagged_rows': count, 'reason': reason, 'action': 'retained_for_review'})
for col in vital_df.columns:
    if col == 'ICUSTAY_ID': continue
    values = final_df[col]
    if not col.startswith(('Temperature_', 'GCS Total_')):
        review(col, values.gt(0) & values.lt(1), 'Sub-unit positive vital reading; possible scaling or recording artifact. No inferred unit correction.')
    if col.startswith('Temperature_'):
        review(col, values.notna() & ~values.between(26,45), 'Outside benchmark VALID temperature interval but within broad extraction bounds; context needed.')
for name, threshold in {'Glucose':2200,'Magnesium':22,'Phosphate':22,'Potassium':15,'Platelet Count':2200,'pO2':770,'INR(PT)':50,'Calcium, Total':20}.items():
    first = final_df[name+'_first']
    last = first + final_df[name+'_change']
    review(name+' first/derived last', first.gt(threshold) | last.gt(threshold), f'Extreme lab endpoint above review threshold {threshold}; thresholds are screening flags, not proof of error.')
for name in ['Creatinine','Urea Nitrogen','White Blood Cells']:
    first=final_df[name+'_first']; last=first+final_df[name+'_change']
    review(name+' first/derived last',first.eq(0)|last.eq(0),'Recorded zero may reflect assay/reporting resolution; retained without arbitrary imputation.')
review('Anion Gap_first',final_df['Anion Gap_first'].lt(0),'Negative anion gap is possible; verify specimen/context rather than blanket deletion.')
review('urine_output_24h',final_df.urine_output_24h.gt(20000),'Above 20 L/day; max 98,280 mL is a sum of multiple high raw entries, not a join duplicate. Verify measurement/irrigation context.')
review('ebl_24h',final_df.ebl_24h.gt(10000),'EBL tails retained by instruction; review during modeling. No cap/removal.')
review('admission_location',final_df.admission_location.eq('** INFO NOT AVAILABLE **'),'Explicit unknown category retained; not silently converted or encoded.')
review_flags=pd.DataFrame(review_rows)
print(review_flags.to_string(index=False))
review_flags.to_csv(report_path/'data_quality_review_flags.csv',index=False)
quality_summary['review_flags'] = review_rows
quality_summary['remaining_review_note'] = 'Structural checks passed, but flagged clinical extremes remain unresolved; no assertion that every retained extreme is physiologically valid.'
quality_summary['corrections'] = {'vital_changed_cells':6840,'lab_changed_cells':6,'invalid_hemoglobin_records':2,'invalid_INR_records':2,'urine_223940':1200,'urine_247320':5215,'EBL_unchanged':True}
(report_path/'data_quality_summary.json').write_text(json.dumps(quality_summary,indent=2)+'\n')


                              column  flagged_rows                                                                                                                            reason              action
                       Mean BP_first             1                             Sub-unit positive vital reading; possible scaling or recording artifact. No inferred unit correction. retained_for_review
                   Systolic BP_first             3                             Sub-unit positive vital reading; possible scaling or recording artifact. No inferred unit correction. retained_for_review
                        Mean BP_last             1                             Sub-unit positive vital reading; possible scaling or recording artifact. No inferred unit correction. retained_for_review
                    Systolic BP_last             1                             Sub-unit positive vital reading; possible scaling or recording artifact. No inferred unit correction. retained_for_re

9674

In [31]:
final_check_df = pd.read_parquet(
    processed_path / "final_dataset.parquet"
)

print("Shape:", final_check_df.shape)
print("Unique ICU stays:", final_check_df["ICUSTAY_ID"].nunique())
print("Duplicates:", final_check_df["ICUSTAY_ID"].duplicated().sum())
print("Missing target:", final_check_df["HOSPITAL_EXPIRE_FLAG"].isna().sum())

Shape: (45253, 85)
Unique ICU stays: 45253
Duplicates: 0
Missing target: 0


In [32]:
numeric_cols = final_check_df.select_dtypes(include="number").columns

negative_counts = {
    col: int((final_check_df[col] < 0).sum())
    for col in numeric_cols
    if not col.endswith("_change")
}

{
    col: count
    for col, count in negative_counts.items()
    if count > 0
}

{'Anion Gap_first': 1}

In [33]:
missing_final = final_check_df.isna().sum()

missing_final[missing_final > 0].sort_values(ascending=False)

Lactate_change      24771
Lactate_first       24771
GCS Total_last      20274
GCS Total_first     20274
GCS Total_min       20274
                    ...  
SpO2_last             726
Heart Rate_last       709
Heart Rate_first      709
Heart Rate_min        709
Heart Rate_max        709
Length: 71, dtype: int64

## Final Dataset Preparation and Quality Control

- Lab, vital, output, intervention, and demographic feature tables were merged using `ICUSTAY_ID`.
- The full cohort of 45,253 ICU stays was preserved with no duplicate IDs.
- Two missingness indicators were added: `lab_missing` and `vital_missing`.
- `HOSPITAL_EXPIRE_FLAG` was added as the mortality target.
- EBL missing values were set to 0 only when no EBL record was present.
- Source-level data quality checks were performed for labs, vitals, urine output, and extreme values.
- Invalid urine and vital measurements were corrected at the source feature level.
- Leakage, duplicate IDs, infinite values, and impossible negative values were checked.
- Remaining lab, vital, and urine missing values were intentionally preserved for later imputation.
- Final dataset: **45,253 rows × 85 columns**.